# soar-lite — the panels, one at a time

Each section runs one panel of the experiment harness and shows the figure it
writes. Run a cell and it re-measures that panel at the budget set in the
cell that defines `MODE`, then draws the figure. The record's own figures
are committed in `docs/figures/`, not stored in this file.

This notebook is the cockpit, not the record: the commands, the output layout
and what the axes mean are in `experiments/README.md`; what each panel measures,
and the caveats behind it, are in `experiments/DESIGN.md`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/watson-tree/soar_lite"

# Colab opens a GitHub notebook inside a checkout at /content/<repo>; a local
# session is already inside one. Either way, work from the repo root.
root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    None,
)
if root is None:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO, "/content/soar_lite"],
        check=True,
    )
    root = Path("/content/soar_lite")
os.chdir(root)
sys.path.insert(0, str(root))  # the harness is a package in the repo, not installed
print(root)

In [ ]:
import importlib.util

# The harness's third-party imports - the `exp` dependency group, which Colab
# happens to carry already. The repo itself is imported from the checkout
# above.
for module in ("jax", "optax", "scipy", "numpy", "matplotlib"):
    if importlib.util.find_spec(module) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", module],
            check=True,
        )

## The budget switch

`MODE` in the next cell is the only knob.

- `"record"` — 20 seeds x 3000 batches, into the harness's own roots. This is
  what produced the figures committed in `docs/figures/`, and it takes hours.
- `"demo"` — 4 seeds x 600 batches, into `results/demo/` and `figures/demo/`.
  Minutes, noisier curves, and it cannot overwrite the record.

One cell ignores the switch: the lr sweep at the bottom writes a committed
input, so it always runs at its own 2 seeds x 3000 batches, and a demo-sized
sweep is refused rather than measured.

On Colab's free tier use `"demo"`: a long run gets disconnected, and the traces
it wrote go with the session.


In [ ]:
from IPython.display import Image, display

from experiments import compare_baselines as harness

MODE = "record"

if MODE == "record":
    SEEDS, MAX_BATCHES, OUT_ARGS = 20, 3000, []
else:
    SEEDS, MAX_BATCHES, OUT_ARGS = 4, 600, ["--out", "demo"]

BUDGET = ["--seeds", str(SEEDS), "--max-batches", str(MAX_BATCHES)]


def show(roots: harness.Roots, relpath: str, width: int = 900) -> None:
    """Display a figure this notebook just wrote, at a readable width."""
    display(Image(filename=str(roots.figures / relpath), width=width))

## Headline: the two real inverse problems

`oscillator_n20_F20` (frequency domain) and `pk_chain_n21_d61` (time domain) are
the problems the library exists for: many coupled parameters in heterogeneous
units, and a loss that costs a simulation.

Read the **batches** axis — that is the wall clock, since one batch is one
parallel pass. `n_paths=16` serves 16 objective points per batch and is drawn
whole here; the same run on the **points** axis (`headline/points.png`) prices
that width instead.

L-BFGS-B is the ceiling reference on this panel, not an opponent: it answers how
close the diagonal secant gets to a full second-order method on a friendly
landscape.


In [ ]:
roots = harness.main(["--groups", "headline", *OUT_ARGS, *BUDGET])
show(roots, "headline/batches.png")

## Smooth: the mechanism, and what it costs

`diag_quad_d100_cond10000` is the control — a diagonal landscape, which is
exactly what the preconditioner estimates, so it should win outright.
`rot_quad_d100_cond1e+06` is the honest boundary: condition number 1e6 with the
rotation off the axis, so the diagonal approximation is approximately right
rather than exact, and it pays for that.

Both are on the one panel, so the price of the diagonal assumption reads as the
gap between them.


In [ ]:
roots = harness.main(["--groups", "smooth", *OUT_ARGS, *BUDGET])
show(roots, "smooth/batches.png")

## Multimodal: the rate, not the floor

`rastrigin_d100` is the rugged landscape — many local basins — and every method
on this panel is first-order. Each stops on its own criterion, so the floor a
curve reaches says more about the stopping rule than about the algorithm: read
the **rate** of the descent, not the last value.

`mu=0.9` is the momentum step. It costs here, which is the other half of the
trade it wins on the rotated quadratic.


In [ ]:
roots = harness.main(["--groups", "multimodal", *OUT_ARGS, *BUDGET])
show(roots, "multimodal/batches.png")

## Invariance: one physics in two unit systems, no re-tune

The same problem with every parameter scaled by a constant. soar-lite's
per-coordinate secant should land on the same outcome, because it estimates
units as it goes; Adam's single learning rate should separate, because its
effective step rescales with the unit.

This panel writes no trace files — there is nothing to re-plot — so the cell
below re-runs it every time. It is cheap.


In [ ]:
roots = harness.main(["--groups", "invariance", *OUT_ARGS, *BUDGET])
show(roots, "invariance/invariance.png")

## Appendix: the Adam learning-rate grid

Adam pays a per-problem learning rate, chosen by an offline sweep; soar-lite does
not. This draws that sweep — how far the optimum drifts across problems, and what
a wrong pick costs.

It is eight learning rates x five problems, so it keeps its own budget — 2 seeds
x 3000 batches — whatever `MODE` says. At 20 seeds this would be ten times the
compute for a plot whose job is to show drift, not to measure a floor. The sweep
also writes `experiments/tuning/adam.json`, a committed input the panels read, so
a cheaper sweep is refused rather than measured.


In [ ]:
roots = harness.main(
    # The sweep's budget is its record's, not the switch's: a different one is
    # refused before a point is measured.
    ["--lr-sweep", *OUT_ARGS, "--seeds", "2", "--max-batches", "3000"],
)
show(roots, "lr_sweep/lr_sweep.png")

## Reproducing the record

The cells above run one panel each. The canonical run is a single command over
the same roots:

```bash
uv run --group exp python -m experiments.compare_baselines \
    --seeds 20 --groups headline,smooth,multimodal,invariance --max-batches 3000
```

`--max-batches` is **batches per method**, and lowering it is the honest way to
spend less: the curves get shorter and noisier but the panel still reads — it is
what `MODE = "demo"` does. Do not lower it to a handful of batches, though: an
arm that stops before the first plotted point has no curve to draw, and the
panel names it instead of inventing one.

Where the output goes: `experiments/results/<problem>/<arm-key>.json` keeps the
raw per-seed traces, `experiments/figures/<panel>/` the PNGs. Both are
gitignored. The record's figures are committed in `docs/figures/`, copied
there from `experiments/figures/` by hand — `docs/figures/README.md` has the
command.
